In [ ]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


# Kvasir-VQA x1 — BLIP VQA baseline

Evaluate BLIP VQA (zero-shot) on Kvasir-VQA x1 metadata. Computes BLEU/ROUGE-L on a sample and yes/no accuracy on the yes/no subset.

In [1]:
import os
from pathlib import Path
import json
import random
import contextlib

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import BlipProcessor, BlipForQuestionAnswering
from datasets import load_dataset
import evaluate



2026-01-14 05:43:29.903612: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Paths & config
HF_DATASET = "SimulaMet-HOST/Kvasir-VQA"  # adjust if using a different HF source

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "blip_baseline" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip-vqa-base"
SAMPLE_N = None  # set None for all rows (can be slow)
SAMPLE_YN = None  # yes/no subset sample
BATCH_SIZE = 96  # aggressive GPU batch; auto-reduced on CPU
SEED = 42


def pick_device():
    if not torch.cuda.is_available():
        return torch.device("cpu"), "CUDA not available; using CPU", None
    try:
        cap = torch.cuda.get_device_capability(0)
        gpu_name = torch.cuda.get_device_name(0)
        arch_list = getattr(torch.cuda, "get_arch_list", lambda: [])()
    except Exception as e:
        return torch.device("cpu"), f"CUDA check failed: {e}; using CPU", None

    def parse_arch(arch: str):
        if not arch.startswith("sm_"):
            return None
        num = arch.split("_")[1]
        if not num.isdigit():
            return None
        major = int(num[:-1]) if len(num) > 1 else int(num)
        minor = int(num[-1]) if len(num) >= 2 else 0
        return (major, minor)

    supported_caps = set(filter(None, (parse_arch(a) for a in arch_list)))
    if cap not in supported_caps:
        return torch.device("cpu"), f"GPU {gpu_name} capability {cap} unsupported by this torch build; using CPU", cap
    return torch.device("cuda"), f"CUDA available (compute capability {cap})", cap

DEVICE, DEVICE_REASON, DEVICE_CAP = pick_device()
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)
if GPU_NAME:
    print("GPU:", GPU_NAME, "capability:", DEVICE_CAP)
print(DEVICE_REASON)

if DEVICE.type == "cpu":
    BATCH_SIZE = min(BATCH_SIZE, 4)
    print("CPU mode: reducing batch size to", BATCH_SIZE)
else:
    print("Using GPU batch size", BATCH_SIZE)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True



Data root: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/blip_baseline/out
Device: cuda
GPU: NVIDIA GeForce RTX 3090 capability: (8, 6)
CUDA available (compute capability (8, 6))
Using GPU batch size 96


In [3]:
# Runtime device sanity check
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        cap = torch.cuda.get_device_capability(0)
        name = torch.cuda.get_device_name(0)
        arch_list = getattr(torch.cuda, 'get_arch_list', lambda: [])()
        print('CUDA device:', name, 'capability:', cap)
        print('Build arch list:', arch_list)
    except Exception as e:
        print('CUDA query failed:', e)
print('Selected DEVICE:', DEVICE)


Torch: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3090 capability: (8, 6)
Build arch list: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Selected DEVICE: cuda


In [4]:
# Load metadata
meta = pd.read_csv(META_CSV)
print("Rows:", len(meta))
print(meta.head())

# Use train/val/test if available else full set
def pick_split(df, split_name):
    if "split" in df.columns and split_name in df["split"].unique():
        return df[df["split"] == split_name].reset_index(drop=True)
    return df.reset_index(drop=True)

dev_df = pick_split(meta, "validation")
if len(dev_df) == 0:
    dev_df = pick_split(meta, "test")
if len(dev_df) == 0:
    dev_df = meta.copy()
print("Dev rows:", len(dev_df))


Rows: 58849
   split                     img_id  \
0  train  cla820gl0s3nv071u4fgd7xgq   
1  train  cla820gl0s3nv071u4fgd7xgq   
2  train  cla820gl0s3nv071u4fgd7xgq   
3  train  cla820gl0s3nv071u4fgd7xgq   
4  train  cla820gl0s3nv071u4fgd7xgq   

                                     image_path  \
0  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
1  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
2  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
3  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
4  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   

                                            question              answer  \
0  Are there any abnormalities in the image? Chec...  ulcerative colitis   
1  Are there any anatomical landmarks in the imag...                none   
2  Are there any instruments in the image? Check ...                none   
3                      Have all polyps been removed?        not relevant   
4                    Is this finding easy to detect?              

In [5]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model_kwargs = {}
if DEVICE.type == "cuda":
    model_kwargs["torch_dtype"] = torch.float16
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME, **model_kwargs)
model = model.to(DEVICE)
model.eval()
MODEL_DTYPE = next(model.parameters()).dtype
print("Loaded BLIP on", DEVICE, "dtype", MODEL_DTYPE)



Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded BLIP on cuda dtype torch.float16


In [6]:
# Fallback: load HF dataset and map img_id -> image if local file missing
HF_IMG_MAP = None
HF_DATASET_CACHE = None

def load_fallback_image(img_id: str):
    global HF_IMG_MAP, HF_DATASET_CACHE
    if HF_IMG_MAP is None:
        # try to load from HF
        try:
            ds_all = load_dataset(HF_DATASET)
            # pick first split
            if 'raw' in ds_all:
                ds_use = ds_all['raw']
            elif 'train' in ds_all:
                ds_use = ds_all['train']
            else:
                ds_use = list(ds_all.values())[0]
            HF_DATASET_CACHE = ds_use
            HF_IMG_MAP = {}
            for ex in ds_use:
                iid = ex.get('img_id') or ex.get('image_id') or ex.get('id') or ex.get('filename')
                if iid is None:
                    continue
                iid = str(iid).split('/')[-1].split('.')[0]
                HF_IMG_MAP[iid] = ex.get('image')
            print("Built HF image map:", len(HF_IMG_MAP))
        except Exception as e:
            print("Fallback HF load failed:", e)
            HF_IMG_MAP = {}
    return HF_IMG_MAP.get(str(img_id))


In [7]:
# Image helpers and batched generation

use_autocast = DEVICE.type == "cuda"

def get_image_for_row(row):
    img_path = Path(row["image_path"])
    if img_path.is_file():
        img = Image.open(img_path).convert("RGB")
    else:
        img_id = row.get("img_id") or img_path.stem
        img = load_fallback_image(img_id)
        if img is None:
            raise FileNotFoundError(f"Image not found locally or in HF fallback: {img_path}")
        if img.mode != "RGB":
            img = img.convert("RGB")
    return img


def batch_generate(df, batch_size=8, desc="BLIP"):
    preds = []
    refs = []
    eff_history = []
    oom_backoffs = 0
    total_steps = (len(df) + batch_size - 1) // batch_size
    for start in tqdm(range(0, len(df), batch_size), total=total_steps, desc=desc):
        eff_bs = min(batch_size, len(df) - start)
        while True:
            batch = df.iloc[start:start + eff_bs]
            images = [get_image_for_row(row) for _, row in batch.iterrows()]
            texts = [str(q) for q in batch["question"]]
            inputs = processor(images=images, text=texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
            ctx = torch.autocast(device_type="cuda", dtype=MODEL_DTYPE) if use_autocast else contextlib.nullcontext()
            try:
                with ctx:
                    outputs = model.generate(**inputs, max_new_tokens=20)
            except torch.cuda.OutOfMemoryError:
                oom_backoffs += 1
                if eff_bs == 1:
                    raise
                eff_bs = max(1, eff_bs // 2)
                torch.cuda.empty_cache()
                continue
            decoded = processor.batch_decode(outputs, skip_special_tokens=True)
            preds.extend(decoded)
            refs.extend(batch["answer"].astype(str).tolist())
            eff_history.append(len(batch))
            break
    if len(eff_history) > 0:
        print(f"Batch stats -> min: {min(eff_history)}, max: {max(eff_history)}, mean: {sum(eff_history)/len(eff_history):.2f}, OOM backoffs: {oom_backoffs}")
    return preds, refs


# Metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")



In [8]:
# Run eval on a sample for BLEU/ROUGE

def eval_sample(df, n=None, batch_size=8):
    if n is not None:
        df = df.sample(min(n, len(df)), random_state=SEED).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)
    preds, refs = batch_generate(df, batch_size=batch_size, desc="BLIP eval")
    bleu_refs = [[r] for r in refs]  # sacrebleu expects list-of-list references
    bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
    rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]
    return df, preds, refs, bleu_score, rouge_l

dev_df_sampled, preds, refs, bleu_score, rouge_l = eval_sample(dev_df, n=SAMPLE_N, batch_size=BATCH_SIZE)

dev_df_sampled = dev_df_sampled.copy()
dev_df_sampled["pred_blip"] = preds

print("BLEU:", bleu_score)
print("ROUGE-L:", rouge_l)

# Save predictions
pred_path = OUT_DIR / "predictions_blip_sample.csv"
dev_df_sampled.to_csv(pred_path, index=False)
print("Saved:", pred_path)



BLIP eval:   0%|          | 0/62 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Yes/No subset accuracy
yn_df = dev_df[dev_df["answer"].astype(str).str.lower().isin(["yes", "no"])]
if SAMPLE_YN is not None:
    yn_df = yn_df.sample(min(SAMPLE_YN, len(yn_df)), random_state=SEED).reset_index(drop=True)
else:
    yn_df = yn_df.reset_index(drop=True)

def to_yesno(text):
    t = str(text).lower()
    if "yes" in t:
        return "yes"
    if "no" in t:
        return "no"
    return "no"

yn_preds, yn_refs = batch_generate(yn_df, batch_size=BATCH_SIZE, desc="BLIP yes/no")
y_pred_labels = [to_yesno(p) for p in yn_preds]
y_true = pd.Series(yn_refs).str.lower().tolist()
acc_yn = (pd.Series(y_pred_labels) == pd.Series(y_true)).mean() if len(yn_df) else 0
print("Yes/No accuracy:", acc_yn, "(n=", len(yn_df), ")")

yn_df_out = yn_df.copy()
yn_df_out["pred_blip_yesno"] = y_pred_labels
yn_path = OUT_DIR / "predictions_blip_yesno.csv"
yn_df_out.to_csv(yn_path, index=False)
print("Saved yes/no preds:", yn_path)



## Notes
- Adjust `SAMPLE_N` / `SAMPLE_YN` to control runtime (set to None for full eval).
- BLEU/ROUGE are on the sampled set; yes/no accuracy is on the yes/no subset.
- Use results to decide if fine-tuning or constrained decoding is needed.
